In [19]:
import os
import pandas as pd
import numpy as np
from collections import Counter, defaultdict
from tqdm import tqdm

In [4]:
directory_path_train = '/content/drive/MyDrive/NTU/AIIM/Workshop 3/extracted_files/set-a'
directory_path_test = '/content/drive/MyDrive/NTU/AIIM/Workshop 3/extracted_files/set-b'

train_entries = os.listdir(directory_path_train)
test_entries = os.listdir(directory_path_test)

# Filter for .txt files and get their full paths
train_files = []
for entry in train_entries:
    if entry.endswith('.txt'):
        full_path = os.path.join(directory_path_train, entry)
        train_files.append(full_path)

print(f"Found {len(train_files)} txt files in training set")

test_files = []
for entry in test_entries:
    if entry.endswith('.txt'):
        full_path = os.path.join(directory_path_test, entry)
        test_files.append(full_path)

print(f"Found {len(test_files)} txt files in test set")

Found 4000 txt files in training set
Found 4000 txt files in test set


In [11]:
# Inspecting one of these csv text files
temp_df = pd.read_csv(train_files[0], sep=',')
print(f"Times are of datatype: {type(temp_df["Time"][0])}")
temp_df.head()

Times are of datatype: <class 'str'>


,Time,Parameter,Value
0,00:00,RecordID,137155.0
1,00:00,Age,68.0
2,00:00,DiasABP,52.0
3,00:00,Gender,1.0
4,00:00,Height,182.9


In [20]:
def analyze_variable_presence(train_files):
    """
    Analyze which variables are most present across all ICU data files

    Params: list of .txt file names
    Returns: dictionary of variable information across all files
    """

    # Track variable presence across files
    variable_counts = Counter()  # Total occurrences across all files
    files_with_variable = Counter()  # Number of files containing each variable
    measurements_per_variable = defaultdict(list)  # Number of measurements per file

    # Static variables (recorded once per patient)
    static_vars = {'RecordID', 'Age', 'Gender', 'Height', 'ICUType', 'Weight'}

    print(f"Analyzing {len(train_files)} files...")

    for file_path in tqdm(train_files):
        try:
            # Read the CSV file
            df = pd.read_csv(file_path)

            # Get unique variables in this file
            variables_in_file = df['Parameter'].unique()

            # Count occurrences of each variable
            variable_value_counts = df['Parameter'].value_counts()

            for var in variables_in_file:
                files_with_variable[var] += 1
                variable_counts[var] += variable_value_counts[var]
                measurements_per_variable[var].append(variable_value_counts[var])

        except Exception as e:
            print(f"Error processing {file_path}: {e}")
            continue

    # Calculate statistics
    total_files = len(train_files)

    results = []
    for var in variable_counts.keys():
        results.append({
            'Variable': var,
            'Total_Measurements': variable_counts[var],
            'Files_Present': files_with_variable[var],
            'Presence_Rate': files_with_variable[var] / total_files * 100,
            'Avg_Measurements_Per_File': np.mean(measurements_per_variable[var]),
            'Is_Static': var in static_vars
        })

    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values('Presence_Rate', ascending=False)

    return results_df

def select_relevant_variables(results_df,
                              min_presence_rate,
                              min_measurements):
    """
    Select the variables based on criteria

    Params:
    - results_df: DataFrame from analyze_variable_presence function,
    - min_presence_rate: minimum percentage of files that must contain the variable,
    - min_measurements: minimum average measurements per file

    Returns: A dataframe of filtered variables meeting criteria
    """

    # Separate static and time-varying variables
    static_vars = results_df[results_df['Is_Static']]
    time_varying = results_df[~results_df['Is_Static']]

    # Filter time-varying
    relevant_time_varying = time_varying[
        (time_varying['Presence_Rate'] >= min_presence_rate) &
        (time_varying['Avg_Measurements_Per_File'] >= min_measurements)
    ]

    # Filter static
    relevant_static = static_vars[static_vars['Presence_Rate'] >= min_presence_rate]

    # Combine
    relevant_vars = pd.concat([relevant_static, relevant_time_varying])
    relevant_vars = relevant_vars.sort_values('Presence_Rate', ascending=False)

    return relevant_vars

In [22]:
results_df = analyze_variable_presence(train_files)

print(f"\nTotal unique variables found: {len(results_df)}")
print(f"\nTop 20 variables by presence rate:")
print(results_df[['Variable', 'Presence_Rate', 'Avg_Measurements_Per_File', 'Is_Static']].head(20).to_string(index=False))

# Select relevant variables
relevant_vars = select_relevant_variables(
    results_df,
    min_presence_rate=30.0,
    min_measurements=3
)

# Create list to be used for training
relevant_variable_names = relevant_vars['Variable'].tolist()
print(f"\n Selected {len(relevant_variable_names)} relevant variables")
print(f"Variable names list: {relevant_variable_names}")

Analyzing 4000 files...


 26%|██▌       | 1023/4000 [00:11<00:24, 122.64it/s]

Error processing /content/drive/MyDrive/NTU/AIIM/Workshop 3/extracted_files/set-a/136336.txt: No columns to parse from file


100%|██████████| 4000/4000 [00:33<00:00, 119.37it/s]



Total unique variables found: 42

Top 20 variables by presence rate:
  Variable  Presence_Rate  Avg_Measurements_Per_File  Is_Static
  RecordID         99.975                   1.000000       True
       Age         99.975                   1.000000       True
    Gender         99.975                   1.000000       True
    Height         99.975                   1.000000       True
   ICUType         99.975                   1.000000       True
    Weight         99.975                  32.299075       True
        HR         98.400                  58.048018      False
      Temp         98.375                  21.954765      False
       BUN         98.375                   3.535959      False
Creatinine         98.375                   3.552986      False
       GCS         98.375                  15.641931      False
       HCT         98.375                   4.640661      False
 Platelets         98.275                   3.587382      False
       WBC         98.150         

In [23]:
# Extract prediciton labels
y_train = pd.read_csv("/content/drive/MyDrive/NTU/AIIM/Workshop 3/Outcomes-a.txt")
y_test = pd.read_csv("/content/drive/MyDrive/NTU/AIIM/Workshop 3/Outcomes-b.txt")

counts = y_train["In-hospital_death"].value_counts()
percentages = y_train["In-hospital_death"].value_counts(normalize=True).mul(100)
y_train_sum = pd.concat([counts, percentages], axis=1, keys=['Count', 'Percentage (%)'])
print("Class distribution for y_train")
display(y_train_sum)

counts2 = y_test["In-hospital_death"].value_counts()
percentages2 = y_test["In-hospital_death"].value_counts(normalize=True).mul(100)
y_test_sum = pd.concat([counts2, percentages2], axis=1, keys=['Count', 'Percentage (%)'])
print("Class distribution for y_test")
display(y_test_sum)

Class distribution for y_train


,Count,Percentage (%)
In-hospital_death,,
0,3446,86.15
1,554,13.85


Class distribution for y_test


,Count,Percentage (%)
In-hospital_death,,
0,3432,85.8
1,568,14.2
